# Demo: Training as a Service (Training-aaS)


`Training-aaS` is the concept of training models on the Hafnia platform on large
and _hidden_ datasets. Hidden datasets refers to datasets that can be used for
training, but are not available for download or direct access.

This is a key for the Hafnia platform, as a hidden dataset ensures data
privacy, and allow models to be trained compliantly and ethically by third parties (you).

## Getting started

1. The first step is to access our platform page and "register" to create a user
   - Production: https://hafnia.milestonesys.com/
   - Staging: https://staging.projecthafnia.com
2. Ask Ahmed to be invited to our TaaS-Ready Organization on the platform.

3. To get access to Training-aaS, you will need to enable it:
   - Production: https://hafnia.milestonesys.com/dashboard/training-aas?enable=true
   - Staging: https://staging.projecthafnia.com/dashboard/training-aas?enable=true


## Training-aaS using the Platform Portal only

1. Click "Dashboard"
2. Click "Training-aaS". If the feature is locked check getting started step 3.
3. Press the "New Experiment" in top right corner to start model training.
4. Select "Dataset".
   - To train a Classification model select a classification dataset such as MNIST, "Caltech256" etc.
   - To train an Object Detection model, select a object detection dataset such as "COCO 2017" or "Midwest Vehicle Detection"
5. Fill in the trainer form
   - **Select the "Public Trainer" tab**.
     We should now have two different trainers available - Image Classification Trainer: Use this trainer for training an Image Classification model. This trainer only works for IC datasets. - Object Detection Trainer: Use this trainer for training an Object Detection model. This trainer only works for OD datasets.
   - **Insert command**:
     - Use the default: `python scripts/train.py`. Description is not needed
   - **Select hardware**: Use the "Free Tier"
6. Press "Next"

### Using a Dataset Recipe with Training-aaS

Training a model using a dataset recipe is very similar to above.
The only difference is that you will need to select the "Data Recipes" tab in Step 4.


# Hafnia SDK: Getting access to the Hafnia Platform

1. Create an API KEY for Training aaS.
   - Go to the platform dashboard
     - Production: https://hafnia.milestonesys.com/dashboard
     - Staging: https://staging.projecthafnia.com/dashboard
   - Select "API keys"
   - Create and copy API key for later.
1. From terminal, configure your machine to access Hafnia:
   Use single command (easier)

```bash
# Staging
API_KEY_STAGING="[INSERT_API_KEY_HERE]"
hafnia profile create "${API_KEY_STAGING}" --name staging --api-url https://api.staging.projecthafnia.com

# Production
API_KEY_PROD=[INSERT_API_KEY_HERE]
hafnia profile create "${API_KEY_PROD}" --name production
```

    For staging select https://api.staging.projecthafnia.com

1. Download `mnist` from terminal to verify that your configuration is working.

   ```bash
   hafnia dataset download mnist --force
   ```


## Hafnia Dataset Format

We have developed our own dataset format.
Key characteristics for a hafnia dataset:

- The dataset is standardized meaning that all dataset are formatted in the same way
- Binary data (image, videos) are stored in their original format
- Metadata is stored in a DataFrame using Polars to do both very fast and expressive operations on metadata.
- Project and tasks are stored separately.

For more information on the hafnia dataset checkout https://github.com/milestone-hafnia/hafnia/blob/main/examples/example_hafnia_dataset.py


In [39]:
from hafnia.dataset.hafnia_dataset import HafniaDataset

# Get 'caltech101' sample dataset
custom_dataset = HafniaDataset.from_name_public_dataset("caltech-101")
custom_dataset.print_basic_stats()

Output()

                          Dataset Statistics                          
                                                                      
  Property                                    Value                   
 ──────────────────────────────────────────────────────────────────── 
  Dataset Name                                caltech-101             
                                                                      
  Version                                     1.0.0                   
                                                                      
  Number of samples                           9144                    
                                                                      
  Task: Classification:image_classification   Number of classes: 102

# Training on Custom Data

To train a model on custom data you will need to:

1. Have your custom dataset as a hafnia dataset
   - Use case 1: You dataset is already in a hafnia format. Load the dataset with `HafniaDataset.from_path(PATH_DATASET)`
   - Use case 2: You dataset is in a supported format and can be imported directly. `HafniaDataset.from_yolo_format()`, `HafniaDataset.from_coco_format()` and `HafniaDataset.from_image_classification_folder`, etc.
   - Use case 3: Your dataset is stored in a custom or unsupported dataset format. Create a custom script to convert it to a hafnia dataset. Checkout the example in https://github.com/milestone-hafnia/hafnia/blob/main/examples/example_custom_dataset.py
2. Upload hafnia dataset to our platform
3. Launch a training through the portal by selecting the newly created dataset


## Step 1: Create Hafnia Dataset from Custom Dataset

Simple example where a hafnia dataset is created from an supported yolo format


In [40]:
from pathlib import Path
import uuid

unique_dataset_name = f"custom-dataset-{str(uuid.uuid4())[:8]}"
path_yolo = Path("../tests/data/dataset_formats/format_yolo")
custom_dataset = HafniaDataset.from_yolo_format(path_yolo)
custom_dataset.info.dataset_name = unique_dataset_name
custom_dataset.print_basic_stats()

Output()

Output()

                   Dataset Statistics                    
                                                         
  Property                      Value                    
 ─────────────────────────────────────────────────────── 
  Dataset Name                  custom-dataset-59a28495  
                                                         
  Version                       0.0.0                    
                                                         
  Number of samples             3                        
                                                         
  Task: Bbox:object_detection   Number of classes: 80

## Step 2: Upload Dataset to Platform

To upload the dataset you simple have to run below function


In [48]:
from hafnia.dataset.dataset_names import SampleField


gallery_image_names = [custom_dataset.samples[SampleField.FILE_PATH].str.split("/").list.last().sort()[0]]

# Upload dataset to Hafnia platform
response = custom_dataset.upload_to_platform(
    interactive=False,
    allow_version_overwrite=True,
    gallery_images=gallery_image_names,
)

[00:02:49] Dataset 'custom-dataset-59a28495' already exists on the Hafnia platform.

Output()

[00:02:52] Syncing dataset to S3 bucket 's3://dataset-writeable-0b2a8d9a-d727-435c-b1c0-7c5b52441d42/sample'

           Writing dataset annotations to /tmp/tmp3y187s9k...

           Sync dataset to s3://dataset-writeable-0b2a8d9a-d727-435c-b1c0-7c5b52441d42/sample

...

           - Found that 3 / 3 data files already exist. Meaning 0 data files will be uploaded.                     
           - Will upload 3 metadata files.                                                                         
           - Total files to upload: 3

           - WARNING: Metadata files for dataset version '0.0.0' already exist. Version will be overwritten as     
           'allow_version_overwrite=True' is set.

Output()

           - Synced dataset in 0.77 seconds.

[00:02:55] Exporting dataset details to platform. This may take up to 30 seconds...

## Step 3: Launch new Training

You will now be able to launch a new training using your new dataset.

_Note: You will need a bigger dataset than provided above to do meaningful trainings_

To launch you actually have two options:

1. Launch the training using the web portal as described earlier in this guide.

2. Another option is to launch an experiment directly using the Hafnia CLI

```bash
DATASET_NAME=[INSERT_DATASET_NAME]
TRAINER_ID=[INSERT_TRAINER_ID]
hafnia experiment create --name "Training on Custom Dataset" --trainer-id $TRAINER_ID --dataset $DATASET_NAME
```

To find available trainers or datasets, you can check the portal or use the cli

```bash
# Get available trainers with the CLI tool. Use '--help' for different options. Use '-v PUBLIC' to get public trainers
hafnia trainer ls
hafnia trainer ls -v PUBLIC

# Get available datasets with the CLI tool. Use '--help' for different options
hafnia dataset ls
```


## Dataset Recipes:

For some use cases it is desirable to perform dataset transformations and/or combine multiple datasets
to train your model on more data or for a specific use case.

For TaaS, we have introduced a concept called dataset recipes to do achieve exactly this.
A Dataset Recipe is a description of the dataset you want in a declarative way.
As the name implies, it is not a dataset, but a description of how multiple datasets and dataset
operations are combined into a specific dataset.

Only after the dataset recipe has been built (or cooked) will it result in an actual dataset.
Unlike a real dataset, a dataset recipe can easily be stored, shared, reproduced, and reused in
a different environment without moving any actual data.

We will demonstrate, the flow and how a dataset recipes is created with python code.


### Dataset Recipe: Create a person and vehicle dataset from multiple datasets

The example demonstrates how we combine two datasets to a single dataset with two
class names `Vehicle` and `Person`.
The challenge of combining the `coco-2017` and `midwest-vehicle-detection` is that they
are labelled with a different set of class names.

The first step is to create two recipes for each dataset,
that loads the desired dataset and maps class names to the desired classes.


In [42]:
from hafnia.dataset.dataset_recipe.dataset_recipe import DatasetRecipe


class_mapping_midwest = {
    "Person": "Person",
    "Vehicle.*": "Vehicle",
    "Vehicle.Trailer": "__REMOVE__",
}

recipe_midwest = DatasetRecipe.from_name(name="midwest-vehicle-detection", version="1.0.0").class_mapper(
    class_mapping=class_mapping_midwest, task_name="object_detection"
)


class_mapping_coco = {
    "person": "Person",
    "bicycle": "Vehicle",
    "car": "Vehicle",
    "motorcycle": "Vehicle",
    "bus": "Vehicle",
    "train": "Vehicle",
    "truck": "Vehicle",
}

recipe_coco = DatasetRecipe.from_name(name="coco-2017", version="1.0.0").class_mapper(
    class_mapping=class_mapping_coco, method="remove_undefined", task_name="object_detection"
)

After class remapping we will now merge `recipe_midwest` and `recipe_coco` into a single recipe / dataset
and remove image samples that doesn't include the classes.


In [43]:
# Merge datasets using recipes and select only samples with 'Person' and 'Vehicle' classes
merged_dataset_recipe = DatasetRecipe.from_merge(
    recipe0=recipe_midwest,
    recipe1=recipe_coco,
).select_samples_by_class_name(name=["Person", "Vehicle"], task_name="object_detection")


Before we go any further, we can actually build the recipe locally on the sample dataset
to verify that the recipe is valid and can be build into a dataset


In [44]:
HafniaDataset.from_name("midwest-vehicle-detection", version="1.0.0").print_basic_stats()

[00:02:32] Dataset found locally. Set 'force=True' or add `--force` flag with cli to re-download

           Reading dataset annotations from Parquet file:                                                          
           /home/petemachine/code/hafnia/notebooks/.data/datasets/midwest-vehicle-detection/annotations.parquet

Output()

                           Dataset Statistics                            
                                                                         
  Property                                    Value                      
 ─────────────────────────────────────────────────────────────────────── 
  Dataset Name                                midwest-vehicle-detection  
                                                                         
  Version                                     1.0.0                      
                                                                         
  Number of samples                           214                        
                                                                         
  Task: Bbox:object_detection                 Number of classes: 13      
                                                                         
  Task: Classification:Weather                Number of classes: 2       
                                                                         
  Task: Classification:Surface Conditions     Number of classes: 2       
                                                                         
  Task: Classification:Geographical Context   Number of classes: 5       
                                                                         
  Task: Classification:Time of Day            Number of classes: 8       
                                                                         
  Task: Classification:Camera Motion          Number of classes: 1       
                                                                         
  Task: Classification:Location Continent     Number of classes: 1

In [45]:
dataset = merged_dataset_recipe.build()
dataset.print_stats()

           Dataset found locally. Set 'force=True' or add `--force` flag with cli to re-download

           Reading dataset annotations from Parquet file:                                                          
           /home/petemachine/code/hafnia/notebooks/.data/datasets/midwest-vehicle-detection/annotations.parquet

Output()

           Dataset found locally. Set 'force=True' or add `--force` flag with cli to re-download

           Reading dataset annotations from Parquet file:                                                          
           /home/petemachine/code/hafnia/notebooks/.data/datasets/coco-2017/annotations.parquet

...

           Task 'mask_detection' with primitive 'Bitmask' has been removed during the merge. This happens if the   
           two datasets do not have the same primitives.

           Task 'Surface Conditions' with primitive 'Classification' has been removed during the merge. This       
           happens if the two datasets do not have the same primitives.

                         Dataset Statistics                          
                                                                     
  Property                      Value                                
 ─────────────────────────────────────────────────────────────────── 
  Dataset Name                  midwest-vehicle-detection+coco-2017  
                                                                     
  Version                       0.0.0                                
                                                                     
  Number of samples             330                                  
                                                                     
  Task: Bbox:object_detection   Number of classes: 2

       Dataset Statistics       
                                
  Split        Samples    Bbox  
 ────────────────────────────── 
  All          330        2507  
                                
  Train        286        2011  
                                
  Validation   20         243   
                                
  Test         24         253  

         Class Count for          
     'Bbox/object_detection'      
┏━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━┓
┃ Class Name ┃ Class Idx ┃ Count ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━┩
│ Person     │ 0         │   612 │
│ Vehicle    │ 1         │  1895 │
└────────────┴───────────┴───────┘

You have now verified that the dataset recipe can be build - at least on the sample dataset -
and you now want to use it for training.

To do this would will need to upload it to the platform:


In [46]:
response = merged_dataset_recipe.as_platform_recipe(recipe_name="merged-person-vehicle-detection", overwrite=True)
dataset_recipe_id = response["id"]
print(f"Created recipe with ID: {dataset_recipe_id}")


Created recipe with ID: 8de19f60-8543-4c95-80b3-858e5b9f805f


Done!
The recipe is now available on the Hafnia platform and can be used for model training and shared in your organization.

Similar to previously, it is now possible to run model training using two approaches:

1. Open the Training aaS web portal. Create a New Experiment and select the newly created recipe.
2. Use the hafnia cli to launch the training from terminal

```bash
RECIPE_NAME=[INSERT_RECIPE_NAME]
TRAINER_ID=[INSERT_TRAINER_ID]
hafnia experiment create --name "Training on Custom Dataset Recipe" --trainer-id $TRAINER_ID --dataset-recipe $RECIPE_NAME
```

To find available recipes through the portal or use the CLI

```bash
# Get available trainers with the CLI tool. Use '--help' for different options. Use '-v PUBLIC' to get public trainers
hafnia recipe ls
```


In [47]:
import subprocess


subprocess.run(
    [
        "hafnia",
        "experiment",
        "create",
        "--name",
        "Training on Custom Dataset Recipe",
        "--trainer-id",
        TRAINER_ID,
        "--dataset-recipe",
        dataset_recipe_id,
    ]
)

NameError: name 'TRAINER_ID' is not defined

## Why Dataset Recipes?

For some use cases it is desirable to perform dataset transformations and/or combine multiple datasets
to train your model on more data or for a specific use case.

Traditionally, users can either change the training script with the desired transformation or do dataset
transformations prior to running the training script.

With TaaS, it is also possible to add dataset transformations in the training script. However, this will only
work for a single dataset, but even for a single dataset, adding use case specific transformations to the
training script will make it less reusable for other use cases.

The other option is to do desired dataset transformations and then inject it into the training script.
If you have access to all the data, then this can be achieved with a custom dataset, but this will not work
for hidden datasets, and even for non-hidden datasets, it might not be desirable to manage large datasets locally
to do the transformations.

With dataset recipes, we can combine and transform datasets in a reusable way as light weight objects on the platform
and  
need to manage the data locally and without the need to change the training script.
